In [1]:
import requests
import csv
import time

In [2]:
# List of all 122 NWS WFO identifiers
wfos = [
    'ABQ', 'ABR', 'AFC', 'AFG', 'AJK', 'AKQ', 'ALY', 'AMA', 'APX', 'ARX', 'BGM', 'BIS', 
    'BMX', 'BOI', 'BOU', 'BOX', 'BRO', 'BTV', 'BUF', 'BYZ', 'CAE', 'CAR', 'CHS', 'CLE', 
    'CRP', 'CTP', 'CYS', 'DDC', 'DLH', 'DMX', 'DTX', 'DVN', 'EAX', 'EKA', 'EPZ', 'EWX', 
    'EYW', 'FFC', 'FGF', 'FGZ', 'FSD', 'FWD', 'GGW', 'GJT', 'GLD', 'GRB', 'GRR', 'GSP', 
    'GYX', 'HFO', 'HGX', 'HNX', 'HUN', 'ILM', 'ILN', 'ILX', 'IND', 'IWX', 'JAN', 'JAX', 
    'JKL', 'LBF', 'LCH', 'LIX', 'LKN', 'LMK', 'LOT', 'LSX', 'LUB', 'LWX', 'LZK', 'MAF', 
    'MEG', 'MFL', 'MFR', 'MHX', 'MKX', 'MLB', 'MOB', 'MPX', 'MQT', 'MRX', 'MSO', 'OAX', 
    'OHX', 'OKX', 'OUN', 'PAH', 'PBZ', 'PDT', 'PHI', 'PIH', 'PSR', 'PUB', 'RAH', 'REV', 
    'RIW', 'RLX', 'RNK', 'SEW', 'SGF', 'SGX', 'SHV', 'SJT', 'SJU', 'SLC', 'STO', 'TAE', 
    'TBW', 'TFX', 'TSA', 'TWC', 'UNR', 'VEF'
]

In [3]:
# Complete mapping of NWS WFO identifiers to Office Name and State
WFO_METADATA = {
    "ABQ": {"name": "Albuquerque", "state": "NM"},
    "ABR": {"name": "Aberdeen", "state": "SD"},
    "AFC": {"name": "Anchorage", "state": "AK"},
    "AFG": {"name": "Fairbanks", "state": "AK"},
    "AJK": {"name": "Juneau", "state": "AK"},
    "AKQ": {"name": "Wakefield / Norfolk", "state": "VA"},
    "ALY": {"name": "Albany", "state": "NY"},
    "AMA": {"name": "Amarillo", "state": "TX"},
    "APX": {"name": "Gaylord", "state": "MI"},
    "ARX": {"name": "La Crosse", "state": "WI"},
    "BGM": {"name": "Binghamton", "state": "NY"},
    "BIS": {"name": "Bismarck", "state": "ND"},
    "BMX": {"name": "Birmingham", "state": "AL"},
    "BOI": {"name": "Boise", "state": "ID"},
    "BOU": {"name": "Boulder / Denver", "state": "CO"},
    "BOX": {"name": "Boston / Norton", "state": "MA"},
    "BRO": {"name": "Brownsville", "state": "TX"},
    "BTV": {"name": "Burlington", "state": "VT"},
    "BUF": {"name": "Buffalo", "state": "NY"},
    "BYZ": {"name": "Billings", "state": "MT"},
    "CAE": {"name": "Columbia", "state": "SC"},
    "CAR": {"name": "Caribou", "state": "ME"},
    "CHS": {"name": "Charleston", "state": "SC"},
    "CLE": {"name": "Cleveland", "state": "OH"},
    "CRP": {"name": "Corpus Christi", "state": "TX"},
    "CTP": {"name": "State College", "state": "PA"},
    "CYS": {"name": "Cheyenne", "state": "WY"},
    "DDC": {"name": "Dodge City", "state": "KS"},
    "DLH": {"name": "Duluth", "state": "MN"},
    "DMX": {"name": "Des Moines", "state": "IA"},
    "DTX": {"name": "Detroit / Pontiac", "state": "MI"},
    "DVN": {"name": "Davenport / Quad Cities", "state": "IA"},
    "EAX": {"name": "Kansas City / Pleasant Hill", "state": "MO"},
    "EKA": {"name": "Eureka", "state": "CA"},
    "EPZ": {"name": "El Paso / Santa Teresa", "state": "NM"},
    "EWX": {"name": "Austin / San Antonio", "state": "TX"},
    "EYW": {"name": "Key West", "state": "FL"},
    "FFC": {"name": "Atlanta / Peachtree City", "state": "GA"},
    "FGF": {"name": "Grand Forks", "state": "ND"},
    "FGZ": {"name": "Flagstaff", "state": "AZ"},
    "FSD": {"name": "Sioux Falls", "state": "SD"},
    "FWD": {"name": "Fort Worth / Dallas", "state": "TX"},
    "GGW": {"name": "Glasgow", "state": "MT"},
    "GJT": {"name": "Grand Junction", "state": "CO"},
    "GLD": {"name": "Goodland", "state": "KS"},
    "GRB": {"name": "Green Bay", "state": "WI"},
    "GRR": {"name": "Grand Rapids", "state": "MI"},
    "GSP": {"name": "Greenville / Spartanburg", "state": "SC"},
    "GUM": {"name": "Guam", "state": "GU"},
    "GYX": {"name": "Gray / Portland", "state": "ME"},
    "HFO": {"name": "Honolulu", "state": "HI"},
    "HGX": {"name": "Houston / Galveston", "state": "TX"},
    "HNX": {"name": "San Joaquin Valley / Hanford", "state": "CA"},
    "HUN": {"name": "Huntsville", "state": "AL"},
    "ILM": {"name": "Wilmington", "state": "NC"},
    "ILN": {"name": "Wilmington / Cincinnati", "state": "OH"},
    "ILX": {"name": "Lincoln", "state": "IL"},
    "IND": {"name": "Indianapolis", "state": "IN"},
    "IWX": {"name": "Northern Indiana / Syracuse", "state": "IN"},
    "JAN": {"name": "Jackson", "state": "MS"},
    "JAX": {"name": "Jacksonville", "state": "FL"},
    "JKL": {"name": "Jackson", "state": "KY"},
    "LBF": {"name": "North Platte", "state": "NE"},
    "LCH": {"name": "Lake Charles", "state": "LA"},
    "LIX": {"name": "New Orleans / Slidell", "state": "LA"},
    "LKN": {"name": "Elko", "state": "NV"},
    "LMK": {"name": "Louisville", "state": "KY"},
    "LOT": {"name": "Chicago / Romeoville", "state": "IL"},
    "LSX": {"name": "St. Louis", "state": "MO"},
    "LUB": {"name": "Lubbock", "state": "TX"},
    "LWX": {"name": "Baltimore / Washington", "state": "VA"},
    "LZK": {"name": "Little Rock", "state": "AR"},
    "MAF": {"name": "Midland / Odessa", "state": "TX"},
    "MEG": {"name": "Memphis", "state": "TN"},
    "MFL": {"name": "Miami", "state": "FL"},
    "MFR": {"name": "Medford", "state": "OR"},
    "MHX": {"name": "Morehead City / Newport", "state": "NC"},
    "MKX": {"name": "Milwaukee / Sullivan", "state": "WI"},
    "MLB": {"name": "Melbourne", "state": "FL"},
    "MOB": {"name": "Mobile", "state": "AL"},
    "MPX": {"name": "Twin Cities / Chanhassen", "state": "MN"},
    "MQT": {"name": "Marquette", "state": "MI"},
    "MRX": {"name": "Morristown / Knoxville", "state": "TN"},
    "MSO": {"name": "Missoula", "state": "MT"},
    "OAX": {"name": "Omaha / Valley", "state": "NE"},
    "OHX": {"name": "Nashville", "state": "TN"},
    "OKX": {"name": "New York City / Upton", "state": "NY"},
    "OUN": {"name": "Norman / Oklahoma City", "state": "OK"},
    "PAH": {"name": "Paducah", "state": "KY"},
    "PBZ": {"name": "Pittsburgh", "state": "PA"},
    "PDT": {"name": "Pendleton", "state": "OR"},
    "PHI": {"name": "Mount Holly / Philadelphia", "state": "NJ"},
    "PIH": {"name": "Pocatello / Idaho Falls", "state": "ID"},
    "PPG": {"name": "Pago Pago", "state": "AS"},
    "PSR": {"name": "Phoenix", "state": "AZ"},
    "PUB": {"name": "Pueblo", "state": "CO"},
    "RAH": {"name": "Raleigh", "state": "NC"},
    "REV": {"name": "Reno", "state": "NV"},
    "RIW": {"name": "Riverton", "state": "WY"},
    "RLX": {"name": "Charleston", "state": "WV"},
    "RNK": {"name": "Roanoke", "state": "VA"},
    "SEW": {"name": "Seattle", "state": "WA"},
    "SGF": {"name": "Springfield", "state": "MO"},
    "SGX": {"name": "San Diego", "state": "CA"},
    "SHV": {"name": "Shreveport", "state": "LA"},
    "SJT": {"name": "San Angelo", "state": "TX"},
    "SJU": {"name": "San Juan", "state": "PR"},
    "SLC": {"name": "Salt Lake City", "state": "UT"},
    "STO": {"name": "Sacramento", "state": "CA"},
    "TAE": {"name": "Tallahassee", "state": "FL"},
    "TBW": {"name": "Tampa Bay / Ruskin", "state": "FL"},
    "TFX": {"name": "Great Falls", "state": "MT"},
    "TSA": {"name": "Tulsa", "state": "OK"},
    "TWC": {"name": "Tucson", "state": "AZ"},
    "UNR": {"name": "Rapid City", "state": "SD"},
    "VEF": {"name": "Las Vegas", "state": "NV"},
}

In [ ]:
output_filename = 'NWS_Tornado_Warning_Stats_2016_2026.csv'

with open(output_filename, 'w', newline='') as csvfile:
    writer = None

    for wfo, meta in WFO_METADATA.items():
        print(f"Fetching data for {wfo} ({meta['name']}, {meta['state']})...")
        
        url = (
            f"https://mesonet.agron.iastate.edu/api/1/cow.json?"
            f"wfo={wfo}&begints=2016-01-01T12%3A00%3A00&endts=2026-09-10T12%3A00%3A00"
            f"&hailsize=1&wind=58&lsrbuffer=15&warningbuffer=1&lsrtype=T&phenomena=TO"
        )
        
        try:
            response = requests.get(url, timeout=15)
            response.raise_for_status()
            data = response.json()
            
            stats = data.get('stats')
            if not stats:
                print(f"  -> No stats returned for {wfo}")
                continue
                
            # Combine office metadata with the IEM stats payload
            stats_row = {
                'WFO': wfo,
                'Office Name': meta['name'],
                'State': meta['state'],
                **stats
            }
            
            if writer is None:
                fieldnames = list(stats_row.keys())
                writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
                writer.writeheader()
                
            writer.writerow(stats_row)
            
        except Exception as e:
            print(f"  -> Error fetching {wfo}: {e}")
            
        time.sleep(1)

print(f"\nFinished! Output saved to '{output_filename}'.")

In [ ]:
'''
output_filename = 'NWS_Tornado_Warning_Stats_2016_2026.csv'

with open(output_filename, 'w', newline='') as csvfile:
    writer = None

    for wfo in wfos:
        print(f"Fetching data for WFO: {wfo}...")
        
        # Formatted URL with your specific timeframes and buffers
        url = (
            f"https://mesonet.agron.iastate.edu/api/1/cow.json?"
            f"wfo={wfo}&begints=2016-01-01T12%3A00%3A00&endts=2026-09-10T12%3A00%3A00"
            f"&hailsize=1&wind=58&lsrbuffer=15&warningbuffer=1&lsrtype=T&phenomena=TO"
        )
        
        try:
            response = requests.get(url, timeout=15)
            response.raise_for_status()
            data = response.json()
            
            # Isolate only the stats block, drop the heavy events data
            stats = data.get('stats')
            if not stats:
                print(f"  -> No stats returned for {wfo}")
                continue
                
            # Prepend the WFO identifier to the row data so we know who is who
            stats['WFO'] = wfo
            
            # Dynamically generate headers based on whatever the IEM API returns
            if writer is None:
                # Force 'WFO' to be the first column in the spreadsheet
                fieldnames = ['WFO'] + [k for k in stats.keys() if k != 'WFO']
                writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
                writer.writeheader()
                
            writer.writerow(stats)
            
        except Exception as e:
            print(f"  -> Error fetching {wfo}: {e}")
            
        # 1-second pause to avoid hammering the IEM servers and getting rate-limited
        time.sleep(1)

print(f"\nFinished! Your data has been saved to '{output_filename}'.")
'''